# Border Sentinel — Dataset v3 Expansion & YOLO Model Training

**Objective**: Fast-track custom training for 4 new border surveillance classes (`weapon`, `drone`, `fire`, `smoke`) combined with the existing 9 core production classes.

### Target Class Taxonomy (13 Classes Total)
- **0–8 (Core Production)**: `person`, `car`, `truck`, `bus`, `motorcycle`, `bicycle`, `animal`, `backpack`, `bag`
- **9–12 (Surveillance Expansion)**: `weapon` (9), `drone` (10), `fire` (11), `smoke` (12)

> **Note on VisDrone**: For speed, this first training pass covers only the 4 new classes' source datasets (`weapon`, `drone`, `d_fire`). VisDrone integration is a separate, subsequent optimization pass and is excluded here.

---
## ⚙️ SETTINGS — EDIT THESE (IN NEXT CELL)
Before running, review the tunable parameters in **Cell 2**:
1. `BASE_MODEL`: Defaults to `"yolov8s.pt"` (small) for fastest training iteration (~30-45 min on Colab T4). Change to `"yolov8l.pt"` for a high-accuracy pass if time permits.
2. `EPOCHS`: Set to `45` (recommended 40–50 for convergence with transfer learning).
3. `IMGSZ`: Set to `640` (standard YOLO inference resolution).
4. `BATCH_SIZE`: Set to `16` (well-tolerated by Colab free T4 16GB VRAM for YOLOv8s; reduce to `8` if training larger models or if Out-Of-Memory occurs).
5. `DRIVE_ROOT`: Google Drive persistence path (defaults to `/content/drive/MyDrive/border-sentinel-training`).
6. `REPO_URL`: GitHub repository clone URL (`https://github.com/Abhinay-code-max/SIH-HACKATHON-AI.git`).

In [1]:
# ==============================================================================
# 1. SETTINGS & HYPERPARAMETERS (TUNABLE)
# ==============================================================================

# Base Model Architecture:
# - 'yolov8s.pt': Recommended for fast turnaround (~30-45 mins on Colab T4)
# - 'yolov8l.pt': Higher capacity/accuracy, slower turnaround
BASE_MODEL = "yolov8s.pt"

# Training Epoch Count (40-50 recommended for solid convergence):
EPOCHS = 45

# Input Image Resolution:
IMGSZ = 640

# Batch Size:
# - 16 is optimal for Colab free T4 (16GB VRAM) with yolov8s.
# - If CUDA Out-Of-Memory (OOM) occurs, lower to 8.
BATCH_SIZE = 16

# Google Drive Persistent Root Directory:
# Keeps datasets, merged output, and trained checkpoints across session disconnects.
DRIVE_ROOT = "/content/drive/MyDrive/border-sentinel-training"

# Git Repository Info (for merge script reuse):
REPO_URL = "https://github.com/Abhinay-code-max/SIH-HACKATHON-AI.git"
REPO_BRANCH = "main"

# Run name following models/registry convention:
MODEL_VERSION = f"YOLO-{'S' if 'yolov8s' in BASE_MODEL else 'L'}-v003-new-classes"

print("=" * 65)
print("BORDER SENTINEL — RUN CONFIGURATION:")
print(f"  Base Model Architecture: {BASE_MODEL}")
print(f"  Model Version / Run ID:  {MODEL_VERSION}")
print(f"  Epochs:                  {EPOCHS}")
print(f"  Image Size:              {IMGSZ}x{IMGSZ}")
print(f"  Batch Size:              {BATCH_SIZE}")
print(f"  Drive Persistence Path:  {DRIVE_ROOT}")
print(f"  Repo Clone Source:       {REPO_URL} [{REPO_BRANCH}]")
print("=" * 65)


BORDER SENTINEL — RUN CONFIGURATION:
  Base Model Architecture: yolov8s.pt
  Model Version / Run ID:  YOLO-S-v003-new-classes
  Epochs:                  45
  Image Size:              640x640
  Batch Size:              16
  Drive Persistence Path:  /content/drive/MyDrive/border-sentinel-training
  Repo Clone Source:       https://github.com/Abhinay-code-max/SIH-HACKATHON-AI.git [main]


In [3]:
# ==============================================================================
# 3. MOUNT GOOGLE DRIVE & INITIALIZE DIRECTORIES
# ==============================================================================

from google.colab import drive

# Mount Google Drive to persist downloaded data and weights across disconnects
drive.mount('/content/drive')

drive_path = Path(DRIVE_ROOT)
raw_dir = drive_path / "raw"
weapon_dir = raw_dir / "weapon"
drone_dir = raw_dir / "drone"
dfire_dir = raw_dir / "d_fire"
staging_dir = drive_path / "dataset_v3_staging"
runs_dir = drive_path / "runs"

for d in [raw_dir, weapon_dir, drone_dir, dfire_dir, staging_dir, runs_dir]:
    d.mkdir(parents=True, exist_ok=True)

print(f"[OK] Drive storage initialized at: {drive_path}")
print(f"  - Raw Datasets:  {raw_dir}")
print(f"  - Merged Staging:{staging_dir}")
print(f"  - Checkpoints:   {runs_dir}")


Mounted at /content/drive
[OK] Drive storage initialized at: /content/drive/MyDrive/border-sentinel-training
  - Raw Datasets:  /content/drive/MyDrive/border-sentinel-training/raw
  - Merged Staging:/content/drive/MyDrive/border-sentinel-training/dataset_v3_staging
  - Checkpoints:   /content/drive/MyDrive/border-sentinel-training/runs


In [2]:
# ==============================================================================
# 2. ENVIRONMENT SETUP & GPU ACCELERATION CHECK
# ==============================================================================

# Install required packages quietly
!pip install -q ultralytics roboflow kaggle pyyaml opencv-python matplotlib

import os
import sys
import torch
from pathlib import Path

print(f"Python Version:  {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")

# Verify GPU is assigned
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"\n[SUCCESS] GPU Detected: {device_name} ({vram_gb:.2f} GB VRAM)")
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
else:
    print("\n[WARNING] NO GPU DETECTED! Training will be painfully slow on CPU.")
    print("--> In Colab menu: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 133.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 6.9 MB/s eta 0:00:00
Python Version:  3.13.15
PyTorch Version: 2.11.0+cu128

[SUCCESS] GPU Detected: Tesla T4 (14.56 GB VRAM)
Tesla T4, 15360 MiB, 14910 MiB


### 🔑 Adding Credentials to Colab Secrets (Left Sidebar)
Before running the next cell, add your API keys securely into Colab's built-in Secrets manager:
1. Click the **Key icon** (Secrets) in Colab's left sidebar.
2. Click **"+ Add new secret"** and add the following 3 entries:
   - Name: `ROBOFLOW_API_KEY` | Value: *Your Roboflow API key* (from roboflow.com -> Account Settings -> Roboflow API)
   - Name: `KAGGLE_USERNAME`  | Value: *Your Kaggle username*
   - Name: `KAGGLE_KEY`       | Value: *Your Kaggle API token key* (from kaggle.com -> Account -> Create New Token -> inside kaggle.json)
3. Toggle the **"Notebook access"** switch to **ON** for all three secrets.

> **Security Guarantee**: Credentials are never typed in plaintext into code cells or committed to Git.

In [4]:
# ==============================================================================
# 4. LOAD CREDENTIALS FROM COLAB SECRETS
# ==============================================================================

from google.colab import userdata

# 1. Roboflow API Key
try:
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
    if not ROBOFLOW_API_KEY:
        raise ValueError("Empty secret")
    print("[OK] ROBOFLOW_API_KEY successfully retrieved from Secrets.")
except Exception as e:
    raise RuntimeError(
        "Missing 'ROBOFLOW_API_KEY' in Colab Secrets!\n"
        "Please click the Key icon in the left sidebar, add 'ROBOFLOW_API_KEY', and grant Notebook access."
    ) from e

# 2. Kaggle Credentials
try:
    KAGGLE_USERNAME = userdata.get('KAGGLE_USERNAME')
    KAGGLE_KEY = userdata.get('KAGGLE_KEY')
    if not KAGGLE_USERNAME or not KAGGLE_KEY:
        raise ValueError("Incomplete credentials")
    os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
    os.environ['KAGGLE_KEY'] = KAGGLE_KEY
    print("[OK] KAGGLE_USERNAME and KAGGLE_KEY successfully loaded into environment.")
except Exception as e:
    raise RuntimeError(
        "Missing 'KAGGLE_USERNAME' or 'KAGGLE_KEY' in Colab Secrets!\n"
        "Please click the Key icon in the left sidebar, add both secrets, and grant Notebook access."
    ) from e


[OK] ROBOFLOW_API_KEY successfully retrieved from Secrets.
[OK] KAGGLE_USERNAME and KAGGLE_KEY successfully loaded into environment.


In [5]:
# ==============================================================================
# 5. DOWNLOAD WEAPON DATASET (ROBOFLOW)
# Workspace: yolo-xkggu | Project: guns-mms73
# Target Class: weapon (ID 9)
# ==============================================================================

from roboflow import Roboflow

# Check if already downloaded to Drive
existing_weapon_imgs = list(weapon_dir.rglob("*.jpg")) + list(weapon_dir.rglob("*.png"))
if len(existing_weapon_imgs) > 50:
    print(f"[SKIP] Weapon dataset already present with {len(existing_weapon_imgs)} images in: {weapon_dir}")
else:
    print("[INFO] Connecting to Roboflow and downloading guns-mms73...")
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project_weapon = rf.workspace("yolo-xkggu").project("guns-mms73")
    try:
        versions = project_weapon.versions()
        v_num = versions[-1].version if versions else 1
    except Exception:
        v_num = 1
    print(f"[INFO] Downloading version {v_num} in YOLOv8 format to {weapon_dir}...")
    dataset_weapon = project_weapon.version(v_num).download("yolov8", location=str(weapon_dir))
    existing_weapon_imgs = list(weapon_dir.rglob("*.jpg")) + list(weapon_dir.rglob("*.png"))
    print(f"[SUCCESS] Weapon dataset download complete: {len(existing_weapon_imgs)} images ready.")


[INFO] Connecting to Roboflow and downloading guns-mms73...
loading Roboflow workspace...
loading Roboflow project...
[INFO] Downloading version 3 in YOLOv8 format to /content/drive/MyDrive/border-sentinel-training/raw/weapon...
[SUCCESS] Weapon dataset download complete: 0 images ready.


In [14]:
import subprocess

print("Roboflow SDK version:", subprocess.run(["pip", "show", "roboflow"], capture_output=True, text=True).stdout.split("\n")[1])

print("\n--- Retrying with explicit object inspection ---")
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project_drone = rf.workspace("keypointdetection-bwrv7").project("drones-detect-qhrmt")
ds = project_drone.version(3).download("yolov8", location=str(local_drone_tmp), overwrite=True)
print("Returned object type:", type(ds))
print("Returned .location:", getattr(ds, "location", "NO .location ATTR"))

print("\n--- Broadest possible search (any file, any extension, whole /content) ---")
result = subprocess.run(["find", "/content", "-newer", "/content/sample_data", "-type", "f"], capture_output=True, text=True)
print(result.stdout[:3000] if result.stdout else "Nothing found matching 'newer than sample_data' anywhere in /content.")

Roboflow SDK version: Version: 1.4.2

--- Retrying with explicit object inspection ---
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/drone_dl_tmp in yolov8:: 100%|██████████| 14415/14415 [00:03<00:00, 3719.69it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Returned object type: <class 'roboflow.core.dataset.Dataset'>
Returned .location: /content/drone_dl_tmp

--- Broadest possible search (any file, any extension, whole /content) ---
/content/.config/gce
/content/drive/MyDrive/Colab Notebooks/Untitled11.ipynb
/content/drive/MyDrive/Mathematical Foundations of Computer Science Examination Papers.gdoc
/content/drive/MyDrive/IMG_20260904_182638 (1).jpg
/content/drive/MyDrive/IMG_20260904_182637.jpg
/content/drive/MyDrive/IMG_20260904_182636 (1).jpg
/content/drive/MyDrive/IMG_20260904_182640.jpg
/content/drive/MyDrive/IMG_20260904_182638.jpg
/content/drive/MyDrive/IMG_20260904_182623.jpg
/content/drive/MyDrive/IMG_20260904_182612.jpg
/con

In [17]:
merged_yaml_path = staging_dir / "data.yaml"
with open(merged_yaml_path) as f:
    merged_classes = yaml.safe_load(f)
print("Merged data.yaml class list:")
names = merged_classes.get("names", {})
if isinstance(names, list):
    names = {i: n for i, n in enumerate(names)}
for idx, name in sorted(names.items()):
    print(f"  {idx}: {name}")

Merged data.yaml class list:
  0: person
  1: car
  2: truck
  3: bus
  4: motorcycle
  5: bicycle
  6: animal
  7: backpack
  8: bag
  9: weapon
  10: drone
  11: fire
  12: smoke


In [6]:
# ==============================================================================
# 6. DOWNLOAD DRONE DATASET (ROBOFLOW)
# Workspace: keypointdetection-bwrv7 | Project: drones-detect-qhrmt
# Target Class: drone (ID 10)
# ==============================================================================

existing_drone_imgs = list(drone_dir.rglob("*.jpg")) + list(drone_dir.rglob("*.png"))
if len(existing_drone_imgs) > 50:
    print(f"[SKIP] Drone dataset already present with {len(existing_drone_imgs)} images in: {drone_dir}")
else:
    print("[INFO] Connecting to Roboflow and downloading drones-detect-qhrmt...")
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project_drone = rf.workspace("keypointdetection-bwrv7").project("drones-detect-qhrmt")
    try:
        versions = project_drone.versions()
        v_num = versions[-1].version if versions else 1
    except Exception:
        v_num = 1
    print(f"[INFO] Downloading version {v_num} in YOLOv8 format to {drone_dir}...")
    dataset_drone = project_drone.version(v_num).download("yolov8", location=str(drone_dir))
    existing_drone_imgs = list(drone_dir.rglob("*.jpg")) + list(drone_dir.rglob("*.png"))
    print(f"[SUCCESS] Drone dataset download complete: {len(existing_drone_imgs)} images ready.")


[INFO] Connecting to Roboflow and downloading drones-detect-qhrmt...
loading Roboflow workspace...
loading Roboflow project...
[INFO] Downloading version 1 in YOLOv8 format to /content/drive/MyDrive/border-sentinel-training/raw/drone...
[SUCCESS] Drone dataset download complete: 0 images ready.


In [7]:
# ==============================================================================
# 7. DOWNLOAD FIRE / SMOKE DATASET (KAGGLE MIRROR)
# Dataset: sayedgamal99/smoke-fire-detection-yolo
# Target Classes: fire (ID 11), smoke (ID 12)
# ==============================================================================

import subprocess

existing_dfire_imgs = list(dfire_dir.rglob("*.jpg")) + list(dfire_dir.rglob("*.png"))
if len(existing_dfire_imgs) > 50:
    print(f"[SKIP] D-Fire dataset already present with {len(existing_dfire_imgs)} images in: {dfire_dir}")
else:
    print("[INFO] Downloading sayedgamal99/smoke-fire-detection-yolo from Kaggle...")
    cmd = ["kaggle", "datasets", "download", "-d", "sayedgamal99/smoke-fire-detection-yolo", "-p", str(dfire_dir), "--unzip"]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        print(f"[ERROR] Kaggle download stdout: {res.stdout}")
        print(f"[ERROR] Kaggle download stderr: {res.stderr}")
        raise RuntimeError(f"Kaggle dataset download failed with exit code {res.returncode}.")
    print("[SUCCESS] D-Fire dataset downloaded and unzipped.")

# Inspect layout and locate true dataset root (handles unzipping into nested directories)
print("\n" + "=" * 65)
print("INSPECTING D-FIRE DIRECTORY STRUCTURE:")
print("=" * 65)
for root, dirs, files in os.walk(dfire_dir):
    rel_path = Path(root).relative_to(dfire_dir)
    if len(rel_path.parts) <= 2:
        img_cnt = sum(1 for f in files if Path(f).suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"])
        txt_cnt = sum(1 for f in files if f.endswith(".txt"))
        yamls = [f for f in files if f.endswith((".yaml", ".yml"))]
        indent = "  " * len(rel_path.parts)
        print(f"{indent}[Dir] {Path(root).name}/ -> images: {img_cnt}, labels: {txt_cnt}, yaml: {yamls}")

def find_dataset_root(base: Path) -> Path:
    """Find directory containing data.yaml or direct train/val/images folders."""
    candidates = [base] + [p for p in base.iterdir() if p.is_dir()] + [p for p in base.glob("*/*") if p.is_dir()]
    for c in candidates:
        if (c / "data.yaml").is_file() or (c / "images").is_dir() or ((c / "train").is_dir() and (c / "train" / "images").is_dir()):
            return c
    return base

actual_dfire_root = find_dataset_root(dfire_dir)
print("=" * 65)
print(f"[RESOLVED] Actual D-Fire root for merge script: {actual_dfire_root}")
print("=" * 65)


[SKIP] D-Fire dataset already present with 21527 images in: /content/drive/MyDrive/border-sentinel-training/raw/d_fire

INSPECTING D-FIRE DIRECTORY STRUCTURE:
[Dir] d_fire/ -> images: 0, labels: 0, yaml: ['data.yaml']
  [Dir] data/ -> images: 0, labels: 0, yaml: []
    [Dir] test/ -> images: 0, labels: 0, yaml: []
    [Dir] train/ -> images: 0, labels: 0, yaml: []
    [Dir] val/ -> images: 0, labels: 0, yaml: []
[RESOLVED] Actual D-Fire root for merge script: /content/drive/MyDrive/border-sentinel-training/raw/d_fire


In [8]:
# ==============================================================================
# 8. CLONE BORDER SENTINEL REPO (FOR MERGE SCRIPT REUSE)
# Commit f522105 contains the tested merge_datasets.py & merge_config.yaml
# ==============================================================================

repo_dir = Path("/content/SIH-HACKATHON-AI")

if (repo_dir / ".git").is_dir():
    print(f"[INFO] Repository already cloned at {repo_dir}. Pulling latest changes from {REPO_BRANCH}...")
    !cd /content/SIH-HACKATHON-AI && git fetch origin && git checkout {REPO_BRANCH} && git pull origin {REPO_BRANCH}
else:
    print(f"[INFO] Cloning repository: {REPO_URL} (branch: {REPO_BRANCH})...")
    !git clone -b {REPO_BRANCH} {REPO_URL} /content/SIH-HACKATHON-AI

merge_script_path = repo_dir / "dataset" / "tools" / "merge_datasets.py"
base_merge_cfg_path = repo_dir / "dataset" / "tools" / "merge_config.yaml"

assert merge_script_path.is_file(), f"Missing merge_datasets.py at: {merge_script_path}"
assert base_merge_cfg_path.is_file(), f"Missing merge_config.yaml at: {base_merge_cfg_path}"
print("[SUCCESS] Verified merge script and base config exist in cloned repository.")


[INFO] Cloning repository: https://github.com/Abhinay-code-max/SIH-HACKATHON-AI.git (branch: main)...
Cloning into '/content/SIH-HACKATHON-AI'...
remote: Enumerating objects: 1141, done.
remote: Counting objects: 100% (1141/1141), done.
remote: Compressing objects: 100% (659/659), done.
remote: Total 1141 (delta 462), reused 1089 (delta 410), pack-reused 0 (from 0)
Receiving objects: 100% (1141/1141), 35.09 MiB | 32.61 MiB/s, done.
Resolving deltas: 100% (462/462), done.
[SUCCESS] Verified merge script and base config exist in cloned repository.


In [9]:
# ==============================================================================
# 9. PROGRAMMATICALLY CONFIGURE MERGE CONFIG FOR COLAB
# - Enables existing_v2 (core 9 classes from repo to avoid catastrophic forgetting)
# - Enables weapon, drone, d_fire (new surveillance classes)
# - Keeps visdrone and test fixtures disabled
# - Points paths directly to cloned repo and Drive download folders
# ==============================================================================

import yaml

with open(base_merge_cfg_path, "r", encoding="utf-8") as f:
    colab_cfg = yaml.safe_load(f)

# Configure output settings
colab_cfg["settings"]["output_dir"] = str(staging_dir)
colab_cfg["settings"]["copy_mode"] = "copy"  # Reliable across filesystem boundaries
colab_cfg["settings"]["keep_empty_as_background"] = True

# Update datasets configuration
for ds in colab_cfg.get("datasets", []):
    d_name = ds.get("name")
    if d_name == "existing_v2":
        ds["enabled"] = True
        ds["path"] = str(repo_dir / "dataset" / "releases" / "v2")
    elif d_name == "weapon":
        ds["enabled"] = True
        ds["path"] = str(weapon_dir)
    elif d_name == "drone":
        ds["enabled"] = True
        ds["path"] = str(drone_dir)
    elif d_name == "d_fire":
        ds["enabled"] = True
        ds["path"] = str(actual_dfire_root)
    elif d_name == "visdrone":
        ds["enabled"] = False  # VisDrone explicitly stays false in this fast pass
    elif d_name in ("fixture_alpha", "fixture_beta"):
        ds["enabled"] = False  # Local test fixtures disabled

# Write updated configuration to Colab workspace
runtime_cfg_path = Path("/content/merge_config_colab.yaml")
with open(runtime_cfg_path, "w", encoding="utf-8") as f:
    yaml.dump(colab_cfg, f, sort_keys=False)

print("[SUCCESS] Generated Colab runtime configuration:")
print("=" * 65)
print(f"Target Staging Directory: {colab_cfg['settings']['output_dir']}")
print("Datasets to Merge:")
for ds in colab_cfg.get("datasets", []):
    status_str = "ENABLED " if ds.get("enabled") else "DISABLED"
    print(f"  [{status_str}] {ds['name']:15s} -> {ds.get('path', 'N/A')}")
print("=" * 65)


[SUCCESS] Generated Colab runtime configuration:
Target Staging Directory: /content/drive/MyDrive/border-sentinel-training/dataset_v3_staging
Datasets to Merge:
  [ENABLED ] existing_v2     -> /content/SIH-HACKATHON-AI/dataset/releases/v2
  [ENABLED ] weapon          -> /content/drive/MyDrive/border-sentinel-training/raw/weapon
  [ENABLED ] drone           -> /content/drive/MyDrive/border-sentinel-training/raw/drone
  [ENABLED ] d_fire          -> /content/drive/MyDrive/border-sentinel-training/raw/d_fire
  [DISABLED] visdrone        -> data/external/visdrone
  [DISABLED] fixture_alpha   -> tests/fixtures/synthetic_datasets/dataset_alpha
  [DISABLED] fixture_beta    -> tests/fixtures/synthetic_datasets/dataset_beta


In [12]:
# 1. See what versions actually exist and how many images each has
for label, ws, proj in [("WEAPON", "yolo-xkggu", "guns-mms73"), ("DRONE", "keypointdetection-bwrv7", "drones-detect-qhrmt")]:
    print(f"\n{'='*65}\n{label}: {ws}/{proj}\n{'='*65}")
    try:
        p = rf.workspace(ws).project(proj)
        versions = p.versions()
        print(f"Found {len(versions)} version(s):")
        for v in versions:
            print(f"  version {v.version}: {getattr(v, 'images', '?')} images (raw: {v})")
    except Exception as e:
        print(f"[EXCEPTION during .versions()]: {type(e).__name__}: {e}")

# 2. Search everywhere in case download() wrote outside the intended Drive folder
import subprocess
print(f"\n{'='*65}\nSEARCHING /content FOR STRAY DATASET FOLDERS\n{'='*65}")
print(subprocess.run(["find", "/content", "-maxdepth", "2", "-type", "d"], capture_output=True, text=True).stdout)


WEAPON: yolo-xkggu/guns-mms73
loading Roboflow workspace...
loading Roboflow project...
Found 2 version(s):
  version 4: 23130 images (raw: {
  "name": "Guns",
  "type": null,
  "version": "4",
  "augmentation": {
    "image": {
      "versions": 3
    },
    "bbnoise": {
      "percent": 5
    },
    "cutout": {
      "count": 3,
      "percent": 10
    },
    "bbrotate": {
      "degrees": 45
    }
  },
  "created": 1680924555.015,
  "preprocessing": {
    "auto-orient": true,
    "remap": {
      "labels": {
        "0": {
          "override": "gun"
        },
        "gun": {
          "override": "gun"
        },
        "rifle": {
          "override": "gun"
        }
      }
    }
  },
  "splits": {
    "valid": 1547,
    "test": 859,
    "train": 20724
  },
  "workspace": "yolo-xkggu"
})
  version 3: 9200 images (raw: {
  "name": "Guns",
  "type": "object-detection",
  "version": "3",
  "augmentation": {},
  "created": 1680759444.885,
  "preprocessing": {
    "auto-orient": t

In [12]:
# ==============================================================================
# 10. EXECUTE DATASET MERGE TOOL (REAL RUN)
# Runs merge_datasets.py to remap, renumber, and partition into dataset_v3_staging
# Hard stop if any new surveillance class has <100 instances
# ==============================================================================

import json
import subprocess

print("[INFO] Executing merge_datasets.py...")
merge_cmd = [
    sys.executable,
    str(merge_script_path),
    "--config", str(runtime_cfg_path),
    "--output", str(staging_dir),
    "--keep-empty",
]

merge_proc = subprocess.run(merge_cmd, capture_output=True, text=True)
print(merge_proc.stdout)
if merge_proc.returncode != 0:
    print(f"[ERROR] merge_datasets.py stderr:\n{merge_proc.stderr}")
    raise RuntimeError(f"Dataset merge failed with exit code {merge_proc.returncode}")

# Load and inspect generated merge_summary.json
summary_file = staging_dir / "merge_summary.json"
assert summary_file.is_file(), f"Summary file not created at: {summary_file}"
with open(summary_file, "r", encoding="utf-8") as f:
    merge_summary = json.load(f)

print("\n" + "=" * 65)
print("DATASET MERGE AUDIT & CLASS COUNTS:")
print("=" * 65)
print(f"Total Images Merged:        {merge_summary['total_images']}")
print(f"Split Distribution:         {merge_summary['split_distribution']}")
print(f"Hard Negative Images Kept:  {merge_summary['total_negative_images']}")
print(f"Boxes Dropped:              {merge_summary['total_boxes_dropped']}")
print("\nFinal Target Class Instance Counts:")

new_classes_to_check = ["weapon", "drone", "fire", "smoke"]
low_count_alerts = []

for cname, count in merge_summary["final_target_class_counts"].items():
    flag = ""
    if cname in new_classes_to_check:
        if count < 100:
            flag = " [!] CRITICAL ERROR: Under 100 instances!"
            low_count_alerts.append((cname, count))
        else:
            flag = " [OK: Sufficient Data]"
    print(f"  Class {cname:12s}: {count:6d}{flag}")
print("=" * 65)

# HARD STOP: Abort execution immediately if instances are suspiciously low
if low_count_alerts:
    alert_details = ", ".join(f"{cname} ({count} instances)" for cname, count in low_count_alerts)
    error_msg = (
        f"\n[HARD STOP] Insufficient instances (<100) detected for target classes: {alert_details}!\n"
        f"Execution aborted to prevent wasting Colab GPU resources on an incomplete or mis-mapped dataset.\n"
        f"Please verify source dataset download folders and label mappings before training."
    )
    raise RuntimeError(error_msg)
else:
    print("\n[SUCCESS] Class counts validated across all target classes! Proceeding to visual sanity check.")


[INFO] Executing merge_datasets.py...

--- YOLO Dataset Merge Completed Successfully ---
Output Directory: /content/drive/MyDrive/border-sentinel-training/dataset_v3_staging
Total Merged Images: 21602
Split Distribution: {'train': 14174, 'val': 3109, 'test': 4319}
Hard Negatives Retained: 9854
Empty Images Dropped: 0
Boxes Dropped: 0
Target Class Counts: {'person': 90, 'car': 0, 'truck': 0, 'bus': 10, 'motorcycle': 0, 'bicycle': 2, 'animal': 0, 'backpack': 0, 'bag': 0, 'weapon': 0, 'drone': 0, 'fire': 14692, 'smoke': 11865}


DATASET MERGE AUDIT & CLASS COUNTS:
Total Images Merged:        21602
Split Distribution:         {'train': 14174, 'val': 3109, 'test': 4319}
Hard Negative Images Kept:  9854
Boxes Dropped:              0

Final Target Class Instance Counts:
  Class person      :     90
  Class car         :      0
  Class truck       :      0
  Class bus         :     10
  Class motorcycle  :      0
  Class bicycle     :      2
  Class animal      :      0
  Class backpack    :  

RuntimeError: 
[HARD STOP] Insufficient instances (<100) detected for target classes: weapon (0 instances), drone (0 instances)!
Execution aborted to prevent wasting Colab GPU resources on an incomplete or mis-mapped dataset.
Please verify source dataset download folders and label mappings before training.

In [ ]:
# ==============================================================================
# 11. VISUAL SANITY CHECK (INLINE SAMPLE INSPECTION)
# Draws bounding boxes on 6 random training samples to visually confirm remapping
# ==============================================================================

import cv2
import random
import matplotlib.pyplot as plt

train_img_dir = staging_dir / "images" / "train"
train_lbl_dir = staging_dir / "labels" / "train"
merged_yaml = staging_dir / "data.yaml"

with open(merged_yaml, "r", encoding="utf-8") as f:
    m_yaml = yaml.safe_load(f)
classes_map = m_yaml.get("names", {})
if isinstance(classes_map, list):
    classes_map = {i: n for i, n in enumerate(classes_map)}

all_train_imgs = list(train_img_dir.glob("*.jpg")) + list(train_img_dir.glob("*.png"))
# Find images that contain annotations
annotated_imgs = []
for imp in all_train_imgs:
    lbl_p = train_lbl_dir / f"{imp.stem}.txt"
    if lbl_p.is_file() and lbl_p.stat().st_size > 0:
        annotated_imgs.append(imp)

sample_count = min(6, len(annotated_imgs))
samples = random.sample(annotated_imgs, sample_count)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, img_path in enumerate(samples):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape
    lbl_file = train_lbl_dir / f"{img_path.stem}.txt"
    lines = lbl_file.read_text().splitlines()

    for line in lines:
        parts = line.strip().split()
        if len(parts) >= 5:
            cid = int(parts[0])
            xc, yc, bw, bh = map(float, parts[1:5])
            x1 = max(0, int((xc - bw / 2) * w))
            y1 = max(0, int((yc - bh / 2) * h))
            x2 = min(w, int((xc + bw / 2) * w))
            y2 = min(h, int((yc + bh / 2) * h))
            cname = classes_map.get(cid, f"cls_{cid}")
            # Bounding box
            cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
            # Label tag
            cv2.putText(img, f"{cname} ({cid})", (x1, max(20, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

    axes[i].imshow(img)
    axes[i].set_title(f"{img_path.name[:25]}... ({len(lines)} boxes)", fontsize=10)
    axes[i].axis('off')

plt.tight_layout()
plt.show()
print("[OK] Visual sanity check rendered. Verify bounding boxes match target labels above.")


In [ ]:
# ==============================================================================
# 12. TRAIN YOLO CUSTOM MODEL
# Saves checkpoints directly to Drive so disconnects do not lose progress
# ==============================================================================

from ultralytics import YOLO

print(f"[INFO] Initializing base model '{BASE_MODEL}' for fine-tuning...")
model = YOLO(BASE_MODEL)

print("=" * 65)
print(f"LAUNCHING TRAINING RUN: {MODEL_VERSION}")
print(f"  Dataset Config: {staging_dir / 'data.yaml'}")
print(f"  Total Epochs:   {EPOCHS}")
print(f"  Batch Size:     {BATCH_SIZE}")
print(f"  Image Size:     {IMGSZ}")
print(f"  Project Output: {runs_dir}")
print("=" * 65)

results = model.train(
    data=str(staging_dir / "data.yaml"),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH_SIZE,
    project=str(runs_dir),
    name=MODEL_VERSION,
    device=0 if torch.cuda.is_available() else "cpu",
    save=True,
    save_period=5,       # Checkpoint every 5 epochs
    workers=2,
    patience=10,
    exist_ok=True,
)

print("\n[SUCCESS] YOLO training execution finished!")


[INFO] Initializing base model 'yolov8s.pt' for fine-tuning...
LAUNCHING TRAINING RUN: YOLO-S-v003-new-classes
  Dataset Config: /content/drive/MyDrive/border-sentinel-training/dataset_v3_staging/data.yaml
  Total Epochs:   45
  Batch Size:     16
  Image Size:     640
  Project Output: /content/drive/MyDrive/border-sentinel-training/runs
Ultralytics 8.4.142 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/border-sentinel-training/dataset_v3_staging/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, ep

In [ ]:
# ==============================================================================
# 13. POST-TRAINING SUMMARY & WEIGHT VERIFICATION
# ==============================================================================

best_weights = runs_dir / MODEL_VERSION / "weights" / "best.pt"
last_weights = runs_dir / MODEL_VERSION / "weights" / "last.pt"

print("=" * 65)
print("TRAINING COMPLETED — ARTIFACT LOCATION:")
print("=" * 65)
if best_weights.is_file():
    size_mb = best_weights.stat().st_size / (1024 * 1024)
    print(f"[SUCCESS] Best weights saved to Google Drive:")
    print(f"  Path: {best_weights}")
    print(f"  Size: {size_mb:.2f} MB")
else:
    print(f"[WARNING] best.pt not found. Checking last.pt: {last_weights.is_file()}")

# Print metric dictionary
try:
    metrics = results.results_dict
    print("\nValidation Performance Metrics:")
    for k, v in metrics.items():
        print(f"  {k:28s}: {v:.4f}")
except Exception as e:
    print(f"[NOTE] Direct metrics dict reading skipped ({e}). Check run training logs.")
print("=" * 65)


## 🚀 Next Steps: Local Deployment & Evaluation

Now that your checkpoint is trained and safely preserved in Google Drive, follow these steps to integrate it into the local Border Sentinel repository:

### 1. Download Weights to Local Registry
From Google Drive, download `best.pt` located at:
```
border-sentinel-training/runs/YOLO-S-v003-new-classes/weights/best.pt
```
Place it in your local repository at:
```
models/registry/YOLO-S-v003-new-classes/weights/best.pt
```

### 2. Update Model Registry Index
Add a new record in `models/registry/registry_index.json` under `"models"` documenting this run:
```json
{
  "version": "YOLO-S-v003-new-classes",
  "base_model": "yolov8s.pt",
  "dataset_version": "v3-staging",
  "date": "<TIMESTAMP>",
  "weights_path": "models/registry/YOLO-S-v003-new-classes/weights/best.pt",
  "metrics": {
    "mAP50": <MAP50>,
    "mAP50-95": <MAP50_95>,
    "precision": <PRECISION>,
    "recall": <RECALL>
  }
}
```

### 3. Re-run Local Evaluation Suite
Execute the evaluation module against this REAL checkpoint:
```bash
.venv\Scripts\python.exe -m ai.detection.evaluation
```
This will replace placeholder/historical numbers with live, independently verified benchmark evidence for the SIH judges.